# 배터리 결함 YOLOv11-seg 본 학습 — v3.4

데이터: `20260707_데이터전처리_v3.4/` (데이터팀 v3.4 산출물, **YOLO 라벨 이미 생성됨** — labels_det / labels_seg)

이전 노트북과의 차이: v3.4는 JSON→YOLO 변환·CT ROI 크롭이 **데이터팀 쪽에서 완료**됨(labels_seg 포함). 데이터는 ZIP 4종으로 배포되므로, 이 노트북은 trainval ZIP을 풀고 `reports/manifest.csv`의 확정 분할로 Ultralytics 학습 뷰(data.yaml)를 **직접 생성**한다. prepare 스크립트·기존 학습 산출물에 의존하지 않음(매번 fresh).

**반영 항목 (0708.md §6)**
- CT: Option B 확정 — `amp=False, lr0=0.0005, epochs=30, patience=15`, seg 유지(단일 클래스 porosity)
- EXT: imbalance 1순위 = **oversampling(옵션 A)**

**진행 원칙**
1. §2 검증 게이트 통과 필수.
2. CT/EXT 모달 독립 학습.
3. 세션 끊김 대비: 매 에폭 `last.pt` + `epoch{N}.pt` 스냅샷 저장, 재실행 시 자동 resume(유실 시 스냅샷 폴백), 완료 후 `{name}_best_backup.pt` 백업(원본 삭제 안 함). **최종 모델은 files.download로 로컬 저장**(FUSE 100% 보장 아님).

## 0. Setup

In [ ]:
!pip -q install ultralytics pyyaml
!nvidia-smi | head -20

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 데이터 로드 — trainval ZIP 압축 해제 (v3.4)

v3.4는 ZIP 4종(CT/EXT × trainval/test)으로 배포됨. 학습엔 trainval만 필요(test는 최종 평가 전용, 별도 로드). Drive FUSE 병목 회피를 위해 로컬 `/content`로 복사 후 해제. 재실행 시 이미 있으면 스킵.

**디스크**: EXT trainval zip 62GB → 해제 시 로컬 ~140GB 필요(Colab ~200GB). 부족하면 CT 먼저 학습·삭제 후 EXT.

In [ ]:
import os, shutil
from pathlib import Path

DATA_V34   = Path('/content/drive/MyDrive/KT/빅프로젝트/data/20260707_데이터전처리_v3.4')
MANIFEST   = DATA_V34/'reports'/'manifest.csv'
DATA_LOCAL = Path('/content/data'); DATA_LOCAL.mkdir(parents=True, exist_ok=True)
assert DATA_V34.is_dir(),  f'v3.4 폴더 없음: {DATA_V34}'
assert MANIFEST.exists(),  f'manifest 없음: {MANIFEST}'

def view_root(base):  # images/ + labels_seg/ 를 함께 가진 디렉터리 탐색
    for lab in Path(base).rglob('labels_seg'):
        if (lab.parent/'images').is_dir():
            return lab.parent
    return None

def unzip_if_missing(zip_path, out_dir):  # 로컬 복사 후 해제 (idempotent)
    out_dir = Path(out_dir)
    if view_root(out_dir):
        print(f'스킵 (이미 해제됨): {out_dir}'); return
    assert Path(zip_path).exists(), f'ZIP 없음: {zip_path}'
    out_dir.mkdir(parents=True, exist_ok=True)
    local_zip = f'/content/{Path(zip_path).name}'
    !cp "{zip_path}" "{local_zip}"
    !unzip -q "{local_zip}" -d "{out_dir}"
    os.remove(local_zip)
    print(f'해제 완료: {out_dir}')

unzip_if_missing(DATA_V34/'battery_CT_v3_trainval.zip',  DATA_LOCAL/'ct')
unzip_if_missing(DATA_V34/'battery_EXT_v3_trainval.zip', DATA_LOCAL/'ext')

CT_ROOT, EXT_ROOT = view_root(DATA_LOCAL/'ct'), view_root(DATA_LOCAL/'ext')
assert CT_ROOT and EXT_ROOT, f'view_root 실패: CT={CT_ROOT} EXT={EXT_ROOT}'
print('CT :', CT_ROOT)
print('EXT:', EXT_ROOT)
!df -h /content | tail -1

### 1.1 data.yaml 생성 — manifest 기반 학습 뷰 (seg)

`reports/manifest.csv`의 확정 분할로 Ultralytics 뷰를 **심볼릭 링크**로 구성(복사 없음). fold 파일 포맷·prepare 스크립트에 의존하지 않음.
- CT: seg fold_0 = **val=fold_0 / train=fold_1~4** (development 중 `included_seg`)
- EXT: `split_role` train/val (`included_seg`)
- EXT 클래스 id→이름은 단일클래스 샘플에서 직접 도출(하드코딩 안 함)

In [ ]:
!pip -q install pyyaml
import yaml
import pandas as pd

_need = [n for n in ('CT_ROOT', 'EXT_ROOT', 'MANIFEST') if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

WORK = Path('/content/work/datasets'); WORK.mkdir(parents=True, exist_ok=True)
CT_OUT, EXT_OUT = WORK/'ct', WORK/'ext'

mani = pd.read_csv(MANIFEST, dtype=str, keep_default_na=False)
seg  = mani[mani['included_seg'].str.lower() == 'true']
ct   = seg[(seg['modality']=='CT') & (seg['split_role']=='development')]
ct_split  = {'val':   ct[ct['fold_id']=='0'],
             'train': ct[ct['fold_id'].isin(['1','2','3','4'])]}
ext  = seg[seg['modality']=='EXT']
ext_split = {'train': ext[ext['split_role']=='train'],
             'val':   ext[ext['split_role']=='val']}

def build(rows, root, out_root, split):  # images/labels 심볼릭 뷰 구성
    isrc, lsrc = root/'images', root/'labels_seg'
    (out_root/'images'/split).mkdir(parents=True, exist_ok=True)
    (out_root/'labels'/split).mkdir(parents=True, exist_ok=True)
    n = miss = 0
    for img, stem in zip(rows['output_image_name'], rows['output_label_stem']):
        si = isrc/img
        if not si.exists(): miss += 1; continue
        di = out_root/'images'/split/img
        if not di.exists(): os.symlink(si, di)
        sl = lsrc/f'{stem}.txt'                       # 정상(결함0)은 라벨 없음 → 배경
        dl = out_root/'labels'/split/f'{stem}.txt'
        if sl.exists() and not dl.exists(): os.symlink(sl, dl)
        n += 1
    print(f'  {out_root.name}/{split}: {n}장 (누락 {miss})')
    return n

for split, rows in ct_split.items():  build(rows, CT_ROOT,  CT_OUT,  split)
for split, rows in ext_split.items(): build(rows, EXT_ROOT, EXT_OUT, split)

def infer_ext_names(rows, root):  # 단일클래스 seg 라벨에서 id→이름 도출 (하드코딩 회피)
    lsrc = root/'labels_seg'; id2name = {}
    cnt = pd.to_numeric(rows['yolo_seg_instance_count'], errors='coerce').fillna(0)
    for want, col, other in [('Damaged','has_damaged','has_pollution'),
                             ('Pollution','has_pollution','has_damaged')]:
        cand = rows[(rows[col].str.lower()=='true') & (rows[other].str.lower()=='false') & (cnt>0)]
        for stem in cand['output_label_stem']:
            lp = lsrc/f'{stem}.txt'
            if lp.exists() and lp.stat().st_size>0:
                id2name[int(lp.read_text().split()[0])] = want; break
    return [id2name[i] for i in sorted(id2name)]

def write_yaml(out_root, names):
    y = out_root/'data.yaml'
    y.write_text(yaml.safe_dump({'path': str(out_root), 'train': 'images/train',
                                 'val': 'images/val', 'names': names},
                                allow_unicode=True, sort_keys=False), encoding='utf-8')
    return y

ct_yaml   = write_yaml(CT_OUT, ['porosity'])
ext_names = infer_ext_names(ext, EXT_ROOT)
assert set(ext_names) == {'Damaged', 'Pollution'}, f'EXT names 도출 실패: {ext_names}'
ext_yaml  = write_yaml(EXT_OUT, ext_names)
print('CT  yaml:', ct_yaml, '| names=[porosity]')
print('EXT yaml:', ext_yaml, '| names=', ext_names)

## 2. 데이터 검증 게이트 (경량)

v3.4 라벨은 데이터팀 QC 완료(approval.json). 여기선 뷰 구조·개수·seg 라벨 포맷만 확인.

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png'}

_need = [n for n in ('ct_yaml', 'ext_yaml') if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

def base_of(data_yaml):  # data.yaml의 path 기준 디렉터리
    y = yaml.safe_load(Path(data_yaml).read_text(encoding='utf-8'))
    return y, Path(y.get('path', Path(data_yaml).parent))

def list_images(spec, base):  # train/val spec(dir 또는 .txt) → 이미지 경로 리스트
    p = Path(spec)
    if not p.is_absolute(): p = base/spec
    if p.suffix.lower() == '.txt':
        out = []
        for l in p.read_text(encoding='utf-8').splitlines():
            l = l.strip()
            if not l: continue
            q = Path(l); out.append(q if q.is_absolute() else base/q)
        return out
    return [q for q in p.rglob('*') if q.suffix.lower() in IMG_EXTS]

def label_for(img):  # 이미지 경로 → YOLO 라벨(.txt) 경로
    return Path(str(img).replace(os.sep+'images'+os.sep, os.sep+'labels'+os.sep)).with_suffix('.txt')

def check(name, data_yaml):
    y, base = base_of(data_yaml)
    for split in ('train', 'val'):
        imgs = list_images(y[split], base)
        pos = [im for im in imgs if label_for(im).exists() and label_for(im).stat().st_size > 0]
        assert imgs, f'{name}/{split} 이미지 0건'
        assert pos,  f'{name}/{split} positive 라벨 0건'
        # seg 포맷 샘플 검증: cls_id int + 짝수 좌표(폴리곤)
        s = label_for(pos[0]).read_text(encoding='utf-8').split(chr(10))[0].split()
        assert len(s) >= 7 and float(s[0]) == int(float(s[0])) and (len(s)-1) % 2 == 0,             f'{name}/{split} seg 포맷 이상: {s[:5]}'
        print(f'{name}/{split}: imgs={len(imgs)} positive={len(pos)}  names={y["names"]}')

check('CT', ct_yaml)
check('EXT', ext_yaml)
print('검증 통과 — 학습 진입 가능')

## 3. Config

In [ ]:
DRIVE_WORK = Path('/content/drive/MyDrive/KT/빅프로젝트/runs_main')
DRIVE_WORK.mkdir(parents=True, exist_ok=True)

MODEL_SIZE = 'm'
IMGSZ = 1280
BATCH = 12          # OOM이면 8
SEED  = 42
SAVE_PERIOD = 10    # N에폭마다 스냅샷 (세션 끊김 대비)

# CT: amp=False = aspect 1:130에서 FP16 NaN 회피. copy_paste=0.0 = 배경(77%)에도 결함을 붙여넣어
#     true negative를 오염시킨 범인(val/cls_loss 폭발·precision 붕괴). 되살리지 말 것.
CT_CFG = dict(amp=False, lr0=0.0005, epochs=30, patience=15,
              hsv_h=0.0, hsv_s=0.0, hsv_v=0.2, degrees=10, flipud=0.5, copy_paste=0.0)
# EXT: imbalance는 §4 oversampling으로 대응. 조기 수렴(ep8 peak)이라 epochs 50·patience 10.
#      copy_paste 유지 = 배경 23%라 오염 위험 낮음(단 val/cls_loss 폭발 감시).
EXT_CFG = dict(amp=True, lr0=0.001, epochs=50, patience=10, copy_paste=0.3)

print('CT ', CT_CFG)
print('EXT', EXT_CFG)

## 4. EXT oversampling (옵션 A) — imbalance 1순위 (0708.md §6.1)

EXT는 Damaged:Pollution ≈ 1:7.6 (train 인스턴스 28,844 : 218,588). Damaged 포함 이미지를 반복해 train 리스트를 만들고 EXT data.yaml의 train을 그 리스트로 교체한다. `OVERSAMPLE_FACTOR`로 조절.

In [ ]:
OVERSAMPLE_FACTOR = 3   # Damaged 포함 이미지 반복 배수 (1이면 미적용)

_need = [n for n in ('base_of', 'ext_yaml', 'label_for', 'list_images') if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

ext_y, ext_base = base_of(ext_yaml)
ext_names = ext_y['names']
if isinstance(ext_names, dict): ext_names = [ext_names[k] for k in sorted(ext_names)]
dmg = [i for i, n in enumerate(ext_names) if str(n).lower() == 'damaged']
assert dmg, f'Damaged 클래스 없음: {ext_names}'
damaged_id = dmg[0]

def has_class(img, cid):  # 라벨에 특정 class id 포함?
    lp = label_for(img)
    if not lp.exists() or lp.stat().st_size == 0: return False
    return any(ln.split() and int(float(ln.split()[0])) == cid
               for ln in lp.read_text(encoding='utf-8').splitlines() if ln.strip())

if OVERSAMPLE_FACTOR <= 1:
    ext_over_yaml = ext_yaml
    print('OVERSAMPLE_FACTOR<=1 → 원본 yaml 사용')
else:
    train_imgs = list_images(ext_y['train'], ext_base)
    dmg_imgs = [im for im in train_imgs if has_class(im, damaged_id)]
    over_list = [str(im) for im in train_imgs] + [str(im) for im in dmg_imgs]*(OVERSAMPLE_FACTOR-1)

    over_txt = ext_base/'train_oversampled.txt'
    over_txt.write_text('\n'.join(over_list), encoding='utf-8')

    ext_over_yaml = ext_yaml.parent/'data_oversampled.yaml'
    y2 = dict(ext_y); y2['train'] = str(over_txt)
    ext_over_yaml.write_text(yaml.safe_dump(y2, allow_unicode=True, sort_keys=False), encoding='utf-8')
    print(f'train 원본 {len(train_imgs)}장 중 Damaged 포함 {len(dmg_imgs)}장 → x{OVERSAMPLE_FACTOR}')
    print(f'oversampled train 총 {len(over_list)}장  →  {ext_over_yaml}')

## 7. 학습 — YOLOv11-seg baseline (CT / EXT)

**세션 끊김 대비 3중 안전장치**:
1. 매 에폭 `last.pt` + `SAVE_PERIOD`(10)마다 `epoch{N}.pt` **영구 스냅샷** Drive 저장.
2. 재실행 시 자동 resume — last.pt 있으면 그걸로, 유실됐으면 **최신 `epoch*.pt`로 폴백** 이어감.
3. 완료 후 best.pt(없으면 last.pt)를 `{name}_best_backup.pt`로 백업 + 크기검증, **원본 삭제 안 함**(0708 유실사고 반영).

⚠️ Colab FUSE 특성상 **가장 최근 epoch은 100% 보장 아님** → **최종 모델은 반드시 `files.download`로 로컬 저장.** fresh 재학습은 rmtree(단, 백업 확인 후).

In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO

_need = [n for n in ('BATCH', 'DRIVE_WORK', 'IMGSZ', 'MODEL_SIZE', 'SAVE_PERIOD', 'SEED') if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

_AUG_KEYS = ('hsv_h','hsv_s','hsv_v','degrees','flipud','fliplr','copy_paste','translate','mosaic')

def train(name, data_yaml, cfg):  # 모달별 학습 (resume 지원) + 체크포인트 백업(삭제 안 함)
    run_dir = DRIVE_WORK/f'train_{name}_v34'
    last = run_dir/'weights'/'last.pt'
    resume_ck = last if last.exists() else None
    if resume_ck is None:  # last.pt 유실 시 최신 주기 스냅샷(epoch*.pt)으로 폴백
        snaps = list(run_dir.glob('weights/epoch*.pt'))
        if snaps: resume_ck = max(snaps, key=lambda p: p.stat().st_mtime)
    if resume_ck is not None:
        print(f'{name}: {resume_ck.name} 발견 → resume')
        m = YOLO(str(resume_ck)); m.train(resume=True)
    else:
        aug = {k: cfg[k] for k in cfg if k in _AUG_KEYS}
        m = YOLO(f'yolo11{MODEL_SIZE}-seg.pt')
        m.train(
            data=str(data_yaml), imgsz=IMGSZ, batch=BATCH,
            epochs=cfg['epochs'], patience=cfg['patience'], save_period=SAVE_PERIOD,
            optimizer='AdamW', lr0=cfg['lr0'], cos_lr=True, amp=cfg['amp'],
            project=str(DRIVE_WORK), name=f'train_{name}_v34', exist_ok=True, seed=SEED,
            **aug,
        )
    # last.pt 절대 삭제 안 함(유실 사고). best 없으면 last 폴백 + 크기검증
    best = run_dir/'weights'/'best.pt'
    final = best if best.exists() else (last if last.exists() else None)
    assert final is not None, f'{name}: weights에 best/last 둘 다 없음 — 학습 산출물 확인'
    bak = DRIVE_WORK/f'{name}_best_backup.pt'
    shutil.copy(final, bak)
    sz = bak.stat().st_size / 1e6
    assert sz > 1, f'{name}: 백업 {sz:.1f}MB(비정상) — FUSE flush 실패 의심, 재복사 필요'
    print(f'{name}: {final.name} → 백업 {bak} ({sz:.0f}MB)  [원본 last/best 삭제 안 함]')
    print(f'  ⚠️ 최종 안전저장: from google.colab import files; files.download("{bak}")  ← 로컬로 받아둘 것')
    return final

In [ ]:
# 7.1 CT 학습
_need = [n for n in ('CT_CFG', 'ct_yaml', 'train') if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

ct_best = train('ct', ct_yaml, CT_CFG)
print('CT best:', ct_best)

In [ ]:
# 7.2 EXT 학습 (oversampled)
_need = [n for n in ('EXT_CFG', 'ext_over_yaml', 'train') if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

ext_best = train('ext', ext_over_yaml, EXT_CFG)
print('EXT best:', ext_best)

## 9. 결과 확인

In [ ]:
from IPython.display import Image as IPImage, display

_need = [n for n in ('DRIVE_WORK',) if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

for name in ['ct', 'ext']:
    for f in ['results.png', 'confusion_matrix.png', 'PR_curve.png']:
        p = DRIVE_WORK/f'train_{name}_v34'/f
        if p.exists():
            print(f'--- {name}/{f} ---'); display(IPImage(str(p)))

print('\n최종 가중치:')
for name in ['ct', 'ext']:
    w = DRIVE_WORK/f'train_{name}_v34'/'weights'
    if w.exists():
        for f in w.iterdir(): print(' ', f)

## 10. 🎯 SAHI lift 검증 (+ 발표용 시각 산출물) — inference-only, 학습 무관

**프로젝트 생사 결정 셀** (0708.md §6.6). 로드맵의 0.22→0.40 점프가 전부 SAHI 가정이라 실측 필요.
- 지표: **image-level PASS/REJECT**(결함 유무) precision/recall/F1 — 제품이 실제로 하는 판정.
- plain(전체 이미지) vs SAHI(슬라이스) 비교. SAHI가 recall↑·총검출수↑면 효과 확정.
- **`run_sahi(best_pt, data_yaml, tag)` 함수 하나로 EXT/CT 둘 다.** EXT는 지금, CT는 재학습 후.
- **📊 발표용 산출물이 내 드라이브 `.../빅프로젝트/sahi_viz/{tag}/`에 저장됨**:
  (1) 지표 비교 막대그래프 PNG, (2) SAHI가 더 잡은 케이스 plain↔SAHI **나란히 비교 이미지**, (3) 요약 txt.
- 이미지 내 글자는 폰트 문제로 **영문**(Colab 한글폰트 없음). 슬라이드에서 한글 캡션 덧붙이면 됨.
- **점검 참고**: CT는 결함이 세로로 김(aspect 1:130) → `slice_sz`가 너무 작으면 결함이 잘림. CT엔 `slice_sz=1024` 등으로 키워 비교해볼 것. per-class(Damaged/Pollution) 분석은 별도.
- §1.1·§2 먼저 실행돼 있어야 함(ext_yaml/ct_yaml, 헬퍼 재사용).

In [ ]:
!pip -q install sahi
from sahi import AutoDetectionModel
from sahi.predict import get_prediction, get_sliced_prediction
from collections import Counter
import random

_need = [n for n in ('DRIVE_WORK', 'base_of', 'ct_yaml', 'ext_yaml', 'label_for', 'list_images') if n not in globals()]
assert not _need, f'★앞 셀을 먼저 실행하세요 — 없는 변수: {_need}'

SAHI_VIZ_ROOT = Path('/content/drive/MyDrive/KT/빅프로젝트/sahi_viz')  # 발표용 시각 산출물 (내 드라이브)

def run_sahi(best_pt, data_yaml, tag, conf=0.25, slice_sz=640, overlap=0.2,
             n_pos=150, n_neg=150, n_viz=12, seed=42):
    best_pt = Path(best_pt)
    assert best_pt.exists(), f'{tag} best.pt 없음: {best_pt}'
    def mk(mt): return AutoDetectionModel.from_pretrained(model_type=mt, model_path=str(best_pt),
                                                          confidence_threshold=conf, device='cuda:0')
    try:    m = mk('ultralytics')       # 최신 sahi
    except Exception: m = mk('yolov8')  # 구버전 폴백

    _, base = base_of(data_yaml)
    val_imgs = list_images(yaml.safe_load(Path(data_yaml).read_text(encoding='utf-8'))['val'], base)
    gt = lambda im: label_for(im).exists() and label_for(im).stat().st_size > 0
    pos = [im for im in val_imgs if gt(im)]; neg = [im for im in val_imgs if not gt(im)]
    random.seed(seed)
    subset = random.sample(pos, min(n_pos, len(pos))) + random.sample(neg, min(n_neg, len(neg)))
    random.shuffle(subset)
    print(f'[{tag}] 서브셋 양성 {min(n_pos,len(pos))} + 음성 {min(n_neg,len(neg))} = {len(subset)}')

    def predict(im, sliced):  # plain/SAHI 추론 결과 객체
        if sliced:
            return get_sliced_prediction(str(im), m, slice_height=slice_sz, slice_width=slice_sz,
                    overlap_height_ratio=overlap, overlap_width_ratio=overlap, verbose=0)
        return get_prediction(str(im), m)

    stat = {'plain': Counter(), 'sahi': Counter()}; inst = {'plain': 0, 'sahi': 0}
    recs = []
    for i, im in enumerate(subset):
        g = gt(im)
        npl = len(predict(im, False).object_prediction_list)
        nsa = len(predict(im, True).object_prediction_list)
        inst['plain'] += npl; inst['sahi'] += nsa
        recs.append((im, npl, nsa, g))
        for nm, pr in [('plain', npl>0), ('sahi', nsa>0)]:
            stat[nm]['TP' if (pr and g) else 'FP' if pr else 'FN' if g else 'TN'] += 1
        if (i+1) % 50 == 0: print(f'  {i+1}/{len(subset)}')

    def prf(c):
        tp, fp, fn = c['TP'], c['FP'], c['FN']
        p = tp/(tp+fp) if tp+fp else 0.0; r = tp/(tp+fn) if tp+fn else 0.0
        return p, r, (2*p*r/(p+r) if p+r else 0.0)
    res = {nm: prf(stat[nm]) for nm in ('plain', 'sahi')}
    print(f'\n=== [{tag}] 이미지단위 결함검출 (PASS/REJECT) ===')
    print(f'{"방식":6}{"P":>7}{"R":>7}{"F1":>7}   TP/FP/FN/TN    총검출')
    for nm in ('plain', 'sahi'):
        c = stat[nm]; p, r, f = res[nm]
        print(f'{nm:6}{p:7.3f}{r:7.3f}{f:7.3f}   {c["TP"]}/{c["FP"]}/{c["FN"]}/{c["TN"]}    {inst[nm]}')

    # ---------- 발표용 시각 산출물 → 내 드라이브 ----------
    viz = SAHI_VIZ_ROOT/tag; viz.mkdir(parents=True, exist_ok=True)

    # (1) 지표 비교 막대그래프 (영문 라벨: Colab 한글폰트 없음)
    import matplotlib.pyplot as plt
    import numpy as np
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    x = np.arange(3); w = 0.35
    ax1.bar(x-w/2, list(res['plain']), w, label='plain', color='#9aa')
    ax1.bar(x+w/2, list(res['sahi']),  w, label='SAHI',  color='#2a7fd4')
    ax1.set_xticks(x); ax1.set_xticklabels(['Precision', 'Recall', 'F1'])
    ax1.set_ylim(0, 1); ax1.set_title('PASS/REJECT (image-level)'); ax1.legend()
    for j in range(3):
        ax1.text(j-w/2, res['plain'][j]+.02, f'{res["plain"][j]:.2f}', ha='center', fontsize=9)
        ax1.text(j+w/2, res['sahi'][j]+.02,  f'{res["sahi"][j]:.2f}',  ha='center', fontsize=9)
    ax2.bar(['plain', 'SAHI'], [inst['plain'], inst['sahi']], color=['#9aa', '#2a7fd4'])
    ax2.set_title('Total detections')
    for j, v in enumerate([inst['plain'], inst['sahi']]): ax2.text(j, v, str(v), ha='center', va='bottom')
    fig.suptitle(f'{tag}: SAHI effect (plain vs SAHI)  conf={conf} slice={slice_sz}')
    fig.tight_layout()
    chart = viz/f'{tag}_sahi_metrics.png'; fig.savefig(chart, dpi=120, bbox_inches='tight'); plt.close(fig)
    print(f'\n[시각] 지표 차트 → {chart}')

    # (2) SAHI가 더 잡은 케이스 plain↔SAHI 나란히 비교
    from PIL import Image, ImageDraw
    cases = sorted(recs, key=lambda r: r[2]-r[1], reverse=True)[:n_viz]
    tmp = Path('/content/_sahi_tmp'); tmp.mkdir(exist_ok=True)
    def annot(im, sliced, fn):  # 예측 오버레이 PNG 경로
        r = predict(im, sliced)
        try:    r.export_visuals(export_dir=str(tmp), file_name=fn, hide_conf=True)
        except TypeError: r.export_visuals(export_dir=str(tmp), file_name=fn)
        return tmp/f'{fn}.png'
    saved = 0
    for k, (im, npl, nsa, g) in enumerate(cases):
        try:
            a = Image.open(annot(im, False, 'p')).convert('RGB')
            b = Image.open(annot(im, True,  's')).convert('RGB')
            H = min(700, a.height, b.height)
            rz = lambda x: x.resize((max(1, int(x.width*H/x.height)), H))
            a, b = rz(a), rz(b); band = 28
            cv = Image.new('RGB', (a.width+b.width+8, H+band), 'white')
            cv.paste(a, (0, band)); cv.paste(b, (a.width+8, band))
            d = ImageDraw.Draw(cv)
            d.text((4, 7),            f'plain: {npl} det', fill=(0, 0, 0))
            d.text((a.width+12, 7),   f'SAHI: {nsa} det',  fill=(30, 120, 200))
            out = viz/f'{tag}_cmp_{k:02d}_p{npl}_s{nsa}.png'; cv.save(out); saved += 1
        except Exception as e:
            print(f'  viz skip {k}: {e}')
    print(f'[시각] 비교 이미지 {saved}장 → {viz}/')

    # (3) 요약 txt
    (viz/f'{tag}_summary.txt').write_text(
        f'{tag} SAHI 검증 (conf={conf}, slice={slice_sz}, overlap={overlap})\n'
        f'plain  P/R/F1 = {res["plain"][0]:.3f}/{res["plain"][1]:.3f}/{res["plain"][2]:.3f}  총검출 {inst["plain"]}\n'
        f'SAHI   P/R/F1 = {res["sahi"][0]:.3f}/{res["sahi"][1]:.3f}/{res["sahi"][2]:.3f}  총검출 {inst["sahi"]}\n',
        encoding='utf-8')
    print(f'[시각] 요약 → {viz}/{tag}_summary.txt')
    print('\n판정: SAHI가 R↑·총검출↑면 효과 확정(작은 결함 회수). 차이 미미하면 0.50 로드맵 재검토(생사).')
    print('발표엔 metrics 차트 + cmp 비교 이미지 사용.')
    return stat, res, inst

# EXT: ext_yaml = 원본 val(oversampled 아님)
run_sahi(DRIVE_WORK/'train_ext_v34'/'weights'/'best.pt', ext_yaml, 'EXT')

# CT: 세로로 긴 결함이라 slice_sz=1024
run_sahi(DRIVE_WORK/'train_ct_v34'/'weights'/'best.pt', ct_yaml, 'CT', slice_sz=1024)